# [5.4] Mamba State Tracking - Exercises

This notebook is the learner-facing implementation path for the state-tracking section. You will build exact latent-state tasks, fit probes on synthetic hidden states, perform causal interventions, and wrap the tiny Mamba implementation from section 5.3 as a token-level state classifier.

```yaml
gt_tier: GT-1
exercise_id: 5.4-mamba-state-tracking
expected_runtime: 60-90 minutes for CPU exercises; 5-10 minutes for CUDA verification
requires_gpu: true for trained model-organism verification; false for the probe exercises
```

Reading map: review the 5.3 selective-scan recurrence, the Chapter 1 linear-probe material, and the OthelloGPT state-probing exercises before attempting the full GPU path.

Failure modes to watch for: sequence-level labels instead of token-level labels, clipped bracket depths which no longer match token deltas, random token splits instead of held-out position splits, probe directions which raise the target logit without lowering the source logit, and classifier wrappers which return only the final token state.


In [ ]:
import sys
from dataclasses import dataclass
from pathlib import Path

import torch as t
from torch import nn
import torch.nn.functional as F

chapter = "chapter5_modern_architectures"
section = "part4_mamba_state_tracking"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part4_mamba_state_tracking.tests as tests
from arena_ext.mamba import MambaConfig, TinyMambaModel


@dataclass(frozen=True)
class StateTrackingBatch:
    tokens: t.Tensor
    states: t.Tensor
    task: str
    vocab: dict[int, str]


@dataclass(frozen=True)
class LinearProbe:
    weight: t.Tensor
    bias: t.Tensor


@dataclass(frozen=True)
class ProbeReport:
    train_accuracy: float
    test_accuracy: float
    num_train: int
    num_test: int


@dataclass(frozen=True)
class InterventionReport:
    source_prediction: int
    target_prediction: int
    intervened_prediction: int
    target_logit_delta: float
    passed: bool


## Synthetic State-Tracking Tasks

Difficulty: medium. Importance: high. Expected output: both tests below should print a passed message. Common bug: producing one label per sequence rather than one latent-state label per token.


In [ ]:
def generate_parity_task(batch: int, seq_len: int, seed: int = 0) -> StateTrackingBatch:
    raise NotImplementedError()


def generate_bracket_depth_task(
    batch: int,
    seq_len: int,
    *,
    max_depth: int = 4,
    seed: int = 0,
) -> StateTrackingBatch:
    raise NotImplementedError()


tests.test_generate_parity_task_matches_cumulative_xor(generate_parity_task)
tests.test_generate_bracket_depth_task_is_bounded_and_consistent(generate_bracket_depth_task)


## Synthetic Hidden States and Held-Out Positions

Difficulty: medium. Importance: high. Expected output: the one-hot features should decode exactly without noise, and the masks should split early versus later positions. Common bug: using a random token split, which removes the OOD-style pressure from the probe evaluation.


In [ ]:
def one_hot_state_features(
    states: t.Tensor,
    num_states: int | None = None,
    *,
    noise_scale: float = 0.0,
    seed: int = 0,
) -> t.Tensor:
    raise NotImplementedError()


def make_position_split(
    states: t.Tensor,
    *,
    train_fraction: float = 0.5,
) -> tuple[t.Tensor, t.Tensor]:
    raise NotImplementedError()


tests.test_one_hot_state_features_shape_noise_and_reference(one_hot_state_features)
tests.test_make_position_split_masks_are_ordered_and_disjoint(make_position_split)


## Closed-Form Linear Probe

Difficulty: medium. Importance: high. Expected output: the probe should recover noisy one-hot latent states on both train and held-out later positions. Common bug: omitting the bias term or applying the mask after the labels and hidden states have been flattened inconsistently.


In [ ]:
def _flatten_masked(
    hidden_states: t.Tensor,
    labels: t.Tensor,
    mask: t.Tensor | None,
) -> tuple[t.Tensor, t.Tensor]:
    raise NotImplementedError()


def fit_linear_probe(
    hidden_states: t.Tensor,
    labels: t.Tensor,
    *,
    num_classes: int | None = None,
    train_mask: t.Tensor | None = None,
    ridge: float = 1e-3,
) -> LinearProbe:
    raise NotImplementedError()


def probe_logits(hidden_states: t.Tensor, probe: LinearProbe) -> t.Tensor:
    raise NotImplementedError()


def probe_predictions(hidden_states: t.Tensor, probe: LinearProbe) -> t.Tensor:
    raise NotImplementedError()


def probe_accuracy(
    hidden_states: t.Tensor,
    labels: t.Tensor,
    probe: LinearProbe,
    mask: t.Tensor | None = None,
) -> float:
    raise NotImplementedError()


def evaluate_probe_generalization(
    hidden_states: t.Tensor,
    labels: t.Tensor,
    probe: LinearProbe,
    train_mask: t.Tensor,
    test_mask: t.Tensor,
) -> ProbeReport:
    raise NotImplementedError()


tests.test_fit_linear_probe_recovers_held_out_one_hot_states(
    fit_linear_probe,
    one_hot_state_features,
    make_position_split,
    evaluate_probe_generalization,
    probe_predictions,
)


## Probe-Derived Causal Intervention

Difficulty: medium. Importance: high. Expected output: the target-minus-source direction should flip the decoded state, while the small matched random direction should not. Common bug: adding only the target class vector instead of using a contrastive target-minus-source direction.


In [ ]:
def state_intervention_direction(
    probe: LinearProbe,
    source_state: int,
    target_state: int,
) -> t.Tensor:
    raise NotImplementedError()


def apply_state_intervention(
    hidden_state: t.Tensor,
    probe: LinearProbe,
    *,
    source_state: int,
    target_state: int,
    coefficient: float = 1.0,
) -> t.Tensor:
    raise NotImplementedError()


def intervention_report(
    hidden_state: t.Tensor,
    probe: LinearProbe,
    *,
    source_state: int,
    target_state: int,
    coefficient: float = 1.0,
) -> InterventionReport:
    raise NotImplementedError()


def random_direction_control(
    hidden_state: t.Tensor,
    probe: LinearProbe,
    *,
    target_state: int,
    coefficient: float = 1.0,
    seed: int = 0,
) -> int:
    raise NotImplementedError()


tests.test_probe_intervention_flips_decoded_state_with_random_control(
    intervention_report,
    random_direction_control,
)


## Tiny Mamba State Classifier

Difficulty: medium. Importance: medium. Expected output: the wrapper should return one logit vector and one hidden state per input token. Common bug: returning only the final token hidden state, which removes the token-level probe surface.


In [ ]:
class TinyMambaStateClassifier(nn.Module):
    def __init__(self, num_states: int = 4):
        super().__init__()
        config = MambaConfig(
            vocab_size=2,
            d_model=32,
            d_inner=64,
            d_state=8,
            d_conv=3,
            dt_rank=4,
            num_layers=1,
            tie_word_embeddings=False,
        )
        self.backbone = TinyMambaModel(config)
        self.head = nn.Linear(config.d_model, num_states)

    def encode(self, input_ids: t.Tensor) -> t.Tensor:
        raise NotImplementedError()

    def forward(
        self,
        input_ids: t.Tensor,
        *,
        return_hidden_states: bool = False,
    ) -> t.Tensor | tuple[t.Tensor, t.Tensor]:
        raise NotImplementedError()


tests.test_tiny_mamba_state_classifier_forward_shapes(TinyMambaStateClassifier)


## Verification Block

After you complete the local exercises, compare against `solutions.py`. The full CUDA path trains the tiny Mamba and Transformer model organisms, checks random-label controls, runs learned hidden-state interventions, and loads pinned Mamba-130M-HF hidden states. The expected report highlights are: tiny Mamba long accuracy around 0.93, random-label long accuracy around 0.36, Transformer long accuracy around 0.62, learned-intervention success around 0.99, and peak VRAM below 1GB for this verification path.


In [ ]:
# Uncomment after completing the exercises and checking against the local solution file.
# from part4_mamba_state_tracking.solutions import run_smoke_test, run_gpu_test
# run_smoke_test(cpu=True)
# run_gpu_test(max_vram_gb=24.0)


## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
